# Radar simulator v2: generate the data, save the CSV, and see what is in it
Put this notebook in the same folder as `radar_sim_v2.py`, then run the cells from top to bottom.

| Cell | What it does |
|---|---|
| 1 | Settings (how many tracks, where to save) |
| 2 | **Generates the tracks and saves the CSV** |
| 3 | Loads the CSV and adds helper columns (motion state, turn rate) |
| 4 to 12 | Plots. Each one is saved as a PNG in `OUT_DIR/plots` |

Data: noise-free truth sampled every 2.5 s. Missile classes are simple placeholder profiles.

In [ ]:
# ---------------- Cell 1: settings ----------------
import os, time, math, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
warnings.filterwarnings("ignore")
import radar_sim_v2 as S

TRACKS_PER_CLASS = 150          # tracks generated for every class
SEED = 7000                     # change for a different random dataset
OUT_DIR = "radar_sim_v2_output" # CSV and plots are written here
CSV_NAME = "radar_sim_v2_dataset.csv"
SAVE_GZIP = False               # True -> also write a .csv.gz
ROUND_DECIMALS = 3              # rounding keeps the file small (mm / mm/s accuracy)
SPLIT = (0.70, 0.15, 0.15)      # train / val / test, by track
os.makedirs(os.path.join(OUT_DIR, "plots"), exist_ok=True)
CLASSES = S.CLASSES
SHORT = {"CIVILIAN_AIRCRAFT":"Civilian","FIGHTER_AIRCRAFT":"Fighter","HELICOPTER":"Helicopter","DRONE":"Drone","SURFACE_MISSILE":"Surf. missile",
         "AERIAL_MISSILE":"Aerial missile","SKIFF":"Skiff","BOAT":"Boat","SHIP":"Ship"}
COL = {"CIVILIAN_AIRCRAFT":"#1f77b4","FIGHTER_AIRCRAFT":"#d62728","HELICOPTER":"#9467bd","DRONE":"#ff7f0e","SURFACE_MISSILE":"#8c564b",
       "AERIAL_MISSILE":"#e377c2","SKIFF":"#2ca02c","BOAT":"#17becf","SHIP":"#7f7f7f"}
plt.rcParams.update({"figure.dpi":100,"axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.alpha":.25,"font.size":10,"axes.titleweight":"bold"})
def save(fig, name):
    p = os.path.join(OUT_DIR, "plots", name); fig.savefig(p, dpi=140, bbox_inches="tight"); print("saved", p)

In [ ]:
# ---------------- Cell 2: generate the tracks and SAVE THE CSV ----------------
t0 = time.time(); frames = []
n_tr, n_va = int(TRACKS_PER_CLASS*SPLIT[0]), int(TRACKS_PER_CLASS*(SPLIT[0]+SPLIT[1]))
for ci, cls in enumerate(CLASSES):
    for i in range(TRACKS_PER_CLASS):
        df, meta = S.simulate_track(cls, SEED + ci*1000 + i, 10000 + ci*1000 + i)
        df["split"] = "train" if i < n_tr else ("val" if i < n_va else "test")
        frames.append(df)
    print(f"{cls:18s} done   ({time.time()-t0:.0f}s)")
D = pd.concat(frames, ignore_index=True)
cols = S.ICD_COLS + ["time_s","true_class","acting_class","phase","subtype","geometry","disguised","speed","lateral_g","split"]
D = D[cols]
fl = D.select_dtypes("float").columns; D[fl] = D[fl].round(ROUND_DECIMALS)
csv_path = os.path.join(OUT_DIR, CSV_NAME); D.to_csv(csv_path, index=False)
print(f"\nSaved {csv_path}   rows={len(D):,}  tracks={D.track_no.nunique():,}   size={os.path.getsize(csv_path)/1e6:.1f} MB")
if SAVE_GZIP: D.to_csv(csv_path + ".gz", index=False, compression="gzip"); print("also saved", csv_path + ".gz")

In [ ]:
# ---------------- Cell 3: load the CSV and add helper columns ----------------
D = pd.read_csv(os.path.join(OUT_DIR, CSV_NAME))
sp = D.vel_mag.clip(lower=1e-6)
D["alt"] = D.pos_z
D["a_along"] = (D.vel_x*D.accel_x + D.vel_y*D.accel_y + D.vel_z*D.accel_z) / sp
D["turn_deg_s"] = np.degrees((D.vel_x*D.accel_y - D.vel_y*D.accel_x).abs() / (D.vel_x**2 + D.vel_y**2 + 1.0))
def motion_state(r):
    if r.speed < 1.5: return "hover / stationary"
    if r.vel_z > 2.0: return "climbing"
    if r.vel_z < -2.0: return "descending"
    if r.turn_deg_s > 0.6: return "turning"
    if abs(r.a_along)/r.speed > 0.01: return "speed change"
    return "steady cruise"
D["motion"] = [motion_state(r) for r in D[["speed","vel_z","turn_deg_s","a_along"]].itertuples()]
MOTIONS = ["steady cruise","speed change","turning","climbing","descending","hover / stationary"]
MCOL = dict(zip(MOTIONS, ["#4c78a8","#f2cf5b","#e45756","#54a24b","#72b7b2","#b279a2"]))
print(D.groupby("true_class").agg(tracks=("track_no","nunique"), packets=("track_no","size")).T)

## Plot 1: the whole dataset in one glance

In [ ]:
# ---------------- Cell 4: one-glance dashboard ----------------
fig = plt.figure(figsize=(20, 11)); gs = fig.add_gridspec(2, 3, hspace=.55, wspace=.28)
# (1) speed vs altitude map
ax = fig.add_subplot(gs[0, 0])
for c in CLASSES:
    d = D[D.true_class == c].sample(min(2500, (D.true_class == c).sum()), random_state=0)
    ax.scatter(d.speed + 0.5, d.alt + 1, s=5, alpha=.35, color=COL[c], label=SHORT[c])
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("speed (m/s)"); ax.set_ylabel("altitude (m)")
ax.set_title("1. Where each class lives: speed vs altitude"); ax.legend(markerscale=3, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(.5, -.14), frameon=False)
# (2) packet share donut
ax = fig.add_subplot(gs[0, 1]); cnt = D.true_class.value_counts().reindex(CLASSES)
w, _ = ax.pie(cnt, colors=[COL[c] for c in CLASSES], startangle=90, wedgeprops=dict(width=.42, edgecolor="w"))
for wi, c, n in zip(w, CLASSES, cnt):
    a = math.radians((wi.theta1 + wi.theta2)/2)
    if n/cnt.sum() >= .06: ax.annotate(f"{SHORT[c]}\n{n/cnt.sum():.0%}", (math.cos(a)*.79, math.sin(a)*.79), ha="center", va="center", fontsize=7.5, color="w", weight="bold")
ax.legend(w, [f"{SHORT[c]}  {n/cnt.sum():.1%}" for c, n in zip(CLASSES, cnt)], loc="upper center", bbox_to_anchor=(.5, -.02), ncol=3, fontsize=8, frameon=False)
ax.text(0, 0, f"{len(D):,}\npackets\n{D.track_no.nunique():,} tracks", ha="center", va="center", fontsize=12, weight="bold"); ax.set_title("2. Who contributes how much data"); ax.grid(False)
# (3) motion state composition
ax = fig.add_subplot(gs[0, 2]); ms = pd.crosstab(D.true_class, D.motion, normalize="index").reindex(index=CLASSES, columns=MOTIONS).fillna(0)*100
left = np.zeros(len(CLASSES))
for m in MOTIONS: ax.barh([SHORT[c] for c in CLASSES], ms[m], left=left, color=MCOL[m], label=m); left += ms[m].values
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("% of packets"); ax.set_title("3. What the targets are doing"); ax.legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(.5, -.12))
# (4) speed ranges
ax = fig.add_subplot(gs[1, 0]); data = [np.log10(D[D.true_class == c].speed + 1) for c in CLASSES]
bp = ax.boxplot(data, vert=False, patch_artist=True, showfliers=False); ax.set_yticklabels([SHORT[c] for c in CLASSES])
for p, c in zip(bp["boxes"], CLASSES): p.set_facecolor(COL[c]); p.set_alpha(.8)
ax.set_xticks([0, 1, 2, 3]); ax.set_xticklabels(["1", "10", "100", "1000"]); ax.set_xlabel("speed (m/s, log scale)"); ax.invert_yaxis(); ax.set_title("4. Speed range per class")
# (5) hardest turning
ax = fig.add_subplot(gs[1, 1]); g99 = D.groupby("true_class").lateral_g.quantile(.99).reindex(CLASSES)
b = ax.bar([SHORT[c] for c in CLASSES], g99, color=[COL[c] for c in CLASSES]); ax.set_yscale("log"); ax.set_ylabel("sideways g (99th percentile, log)")
for r, v in zip(b, g99): ax.text(r.get_x() + r.get_width()/2, v*1.1, f"{v:.2f}", ha="center", fontsize=8)
plt.setp(ax.get_xticklabels(), rotation=35, ha="right"); ax.set_title("5. How hard each class manoeuvres")
# (6) disguise donut
ax = fig.add_subplot(gs[1, 2]); dis = D[D.disguised].groupby("true_class").track_no.nunique(); tot = D.groupby("true_class").track_no.nunique()
mim = {c: D[(D.true_class == c) & D.disguised & (D.acting_class != c)].acting_class.mode().iloc[0] for c in dis.index}
labels = [f"{SHORT[c]} acting as {SHORT[mim[c]]}" for c in dis.index]
ax.pie(dis, labels=[f"{l}\n{n} tracks" for l, n in zip(labels, dis)], colors=[COL[c] for c in dis.index], startangle=90, wedgeprops=dict(width=.45, edgecolor="w"), textprops=dict(fontsize=8))
ax.text(0, 0, f"{dis.sum()} of {tot.sum()}\ntracks\ndisguised", ha="center", va="center", fontsize=11, weight="bold"); ax.set_title("6. Disguised tracks"); ax.grid(False)
fig.suptitle("Radar simulator v2: the dataset at a glance", fontsize=18, weight="bold", y=.97); save(fig, "01_dataset_at_a_glance.png"); plt.show()

## Plot 2: speed and altitude zones (which classes overlap, and where)

In [ ]:
# ---------------- Cell 5: class zones with density contours ----------------
fig, ax = plt.subplots(figsize=(13, 8)); xs, ys = np.log10(D.speed + 1), np.log10(D.alt + 1)
for c in CLASSES:
    m = (D.true_class == c).values
    H, xe, ye = np.histogram2d(xs[m], ys[m], bins=60, range=[[0, 3.3], [0, 4.3]]); H = H / H.max()
    ax.contourf(0.5*(xe[1:] + xe[:-1]), 0.5*(ye[1:] + ye[:-1]), H.T, levels=[.05, 1], colors=[COL[c]], alpha=.28)
    ax.contour(0.5*(xe[1:] + xe[:-1]), 0.5*(ye[1:] + ye[:-1]), H.T, levels=[.05], colors=[COL[c]], linewidths=1.4)
    ax.text(np.median(xs[m]), np.median(ys[m]), SHORT[c], color=COL[c], weight="bold", ha="center", fontsize=11, bbox=dict(fc="w", ec="none", alpha=.7, pad=1))
ax.set_xticks([0, 1, 2, 3]); ax.set_xticklabels(["1", "10", "100", "1000"]); ax.set_yticks([0, 1, 2, 3, 4]); ax.set_yticklabels(["0", "10", "100", "1 000", "10 000"])
ax.set_xlabel("speed (m/s)"); ax.set_ylabel("altitude (m)"); ax.set_title("Where each class lives (shaded = where most of its packets are)\nOverlaps are the classes that motion alone cannot separate")
save(fig, "02_speed_altitude_zones.png"); plt.show()

## Plot 3: how alike are the classes? (overlap heat-map)

In [ ]:
# ---------------- Cell 6: pairwise overlap of behaviour (a hint of what will be hard to classify) ----------------
feat = np.column_stack([np.log10(D.speed + 1), np.log10(D.alt + 1), np.log10(D.lateral_g + 0.01), np.log10(D.vel_z.abs() + 1)])
edges = [np.linspace(feat[:, k].min(), feat[:, k].max(), 16) for k in range(4)]
P = {}
for c in CLASSES:
    m = (D.true_class == c).values; P[c] = np.mean([np.histogram(feat[m, k], bins=edges[k])[0] / m.sum() for k in range(4)], axis=0)
O = np.array([[np.minimum(P[a], P[b]).sum() for b in CLASSES] for a in CLASSES])
fig, ax = plt.subplots(figsize=(9, 7.5)); im = ax.imshow(O, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(9)); ax.set_xticklabels([SHORT[c] for c in CLASSES], rotation=40, ha="right"); ax.set_yticks(range(9)); ax.set_yticklabels([SHORT[c] for c in CLASSES]); ax.grid(False)
for i in range(9):
    for j in range(9): ax.text(j, i, f"{O[i,j]:.2f}", ha="center", va="center", fontsize=8, color="w" if O[i, j] > .6 else "k")
plt.colorbar(im, label="overlap (0 = never alike, 1 = identical)"); ax.set_title("How alike are the classes? (speed, height, turning g, climb rate)")
save(fig, "03_class_overlap.png"); plt.show()

## Plot 4: speed vs manoeuvring (what each class can do)

In [ ]:
# ---------------- Cell 7: speed vs sideways g ----------------
fig, ax = plt.subplots(figsize=(13, 7.5))
for c in CLASSES:
    d = D[D.true_class == c].sample(min(3000, (D.true_class == c).sum()), random_state=1)
    ax.scatter(d.speed + .5, d.lateral_g + 1e-3, s=5, alpha=.3, color=COL[c], label=SHORT[c])
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("speed (m/s)"); ax.set_ylabel("sideways acceleration (g)")
for g, t in [(.1, "gentle"), (1, "1 g"), (5, "hard")]: ax.axhline(g, color="k", ls=":", lw=.8); ax.text(.6, g*1.1, t, fontsize=8)
ax.legend(markerscale=3, ncol=3); ax.set_title("Speed vs manoeuvring: slow-and-agile (skiffs, drones) to fast-and-hard (fighters, missiles)")
save(fig, "04_speed_vs_turning_g.png"); plt.show()

## Plot 5: what the targets are doing, class by class

In [ ]:
# ---------------- Cell 8: behaviour phases per class (small multiples) ----------------
fig, axs = plt.subplots(3, 3, figsize=(18, 11)); axs = axs.ravel()
for ax, c in zip(axs, CLASSES):
    d = D[D.true_class == c]; ph = d.phase.str.replace("mimic_", "mimic ").value_counts(normalize=True).head(7)[::-1]*100
    ax.barh(ph.index, ph.values, color=COL[c]); ax.set_title(SHORT[c]); ax.set_xlabel("% of packets")
    for y, v in enumerate(ph.values): ax.text(v + .5, y, f"{v:.0f}%", va="center", fontsize=8)
    ax.set_xlim(0, ph.max()*1.2)
fig.suptitle("Behaviour phases: what the generator makes each class do (top phases, share of packets)", fontsize=15, weight="bold"); plt.tight_layout(); save(fig, "05_behaviour_phases.png"); plt.show()

## Plot 6: example tracks (colour = speed)

In [ ]:
# ---------------- Cell 9: gallery of example tracks ----------------
import matplotlib as mpl
fig, axs = plt.subplots(9, 3, figsize=(17, 33))
for i, c in enumerate(CLASSES):
    d = D[D.true_class == c]; tns = d.track_no.unique()[[3, 40, 90]]
    for j, tn in enumerate(tns):
        g = d[d.track_no == tn]; ax = axs[i, j]
        sc = ax.scatter(g.pos_x/1e3, g.pos_y/1e3, c=g.speed, s=8, cmap="viridis"); ax.plot(g.pos_x.iloc[0]/1e3, g.pos_y.iloc[0]/1e3, "r^", ms=8)
        ax.set_aspect("equal", "datalim"); ax.set_title(f"{SHORT[c]} | {g.subtype.iloc[0].replace('_PLACEHOLDER','')}", fontsize=9, color=COL[c]); plt.colorbar(sc, ax=ax, label="speed m/s", shrink=.8)
        ax.set_xlabel("x (km)"); ax.set_ylabel("y (km)")
fig.suptitle("Example tracks seen from above (red triangle = first packet)", fontsize=15, weight="bold", y=.995); plt.tight_layout(); save(fig, "06_example_tracks.png"); plt.show()

## Plot 7: hovering (helicopters and multirotor drones)

In [ ]:
# ---------------- Cell 10: hovering ----------------
fig = plt.figure(figsize=(19, 10)); gs = fig.add_gridspec(2, 3, hspace=.35, wspace=.3)
for k, c in enumerate(["HELICOPTER", "DRONE"]):
    ax = fig.add_subplot(gs[0, k]); ph = D[D.true_class == c].phase.str.replace("mimic_", "mimic ").value_counts(); small = ph[ph/ph.sum() < .03]
    ph = pd.concat([ph[ph/ph.sum() >= .03], pd.Series({"other": small.sum()})]) if len(small) else ph
    w, _ = ax.pie(ph, startangle=90, colors=plt.cm.tab20.colors[:len(ph)], wedgeprops=dict(width=.45, edgecolor="w"))
    ax.legend(w, [f"{n}  {v/ph.sum():.0%}" for n, v in ph.items()], loc="upper center", bbox_to_anchor=(.5, 0.02), ncol=2, fontsize=8, frameon=False); ax.set_title(f"{SHORT[c]}: what it spends its time doing"); ax.grid(False)
ax = fig.add_subplot(gs[0, 2]); hv = D.assign(h=D.speed < 1.5).groupby("true_class").h.mean().reindex(CLASSES)*100
ax.bar([SHORT[c] for c in CLASSES], hv, color=[COL[c] for c in CLASSES]); plt.setp(ax.get_xticklabels(), rotation=35, ha="right"); ax.set_ylabel("% of packets with speed < 1.5 m/s"); ax.set_title("Time at (almost) zero speed\nhover for rotorcraft, slow drift for vessels", fontsize=10)
for x, v in enumerate(hv): ax.text(x, v + .5, f"{v:.0f}%", ha="center", fontsize=8)
def longest_hover(cls, sub=None):
    d = D[(D.true_class == cls)] if sub is None else D[(D.true_class == cls) & (D.subtype == sub)]; best = (0, None, None)
    for tn, g in d.groupby("track_no"):
        h = (g.speed.values < 1.5).astype(int); run = mx = 0; end = 0
        for i, v in enumerate(h):
            run = run + 1 if v else 0
            if run > mx: mx, end = run, i
        if mx > best[0]: best = (mx, tn, end - mx + 1)
    return best
for k, (cls, sub) in enumerate([("HELICOPTER", "observation"), ("DRONE", "multirotor")]):
    n, tn, s0 = longest_hover(cls, sub); g = D[D.track_no == tn].iloc[s0:s0 + n]; ax = fig.add_subplot(gs[1, k])
    sc = ax.scatter(g.pos_x - g.pos_x.mean(), g.pos_y - g.pos_y.mean(), c=g.time_s, cmap="plasma", s=18); ax.plot(g.pos_x - g.pos_x.mean(), g.pos_y - g.pos_y.mean(), lw=.6, color="grey"); ax.set_aspect("equal", "datalim")
    plt.colorbar(sc, ax=ax, label="time (s)"); ax.set_xlabel("east (m)"); ax.set_ylabel("north (m)"); ax.set_title(f"{SHORT[cls]} longest hover: {n*2.5:.0f} s, wanders {np.ptp(g.pos_x):.1f} x {np.ptp(g.pos_y):.1f} m")
ax = fig.add_subplot(gs[1, 2]); n, tn, s0 = longest_hover("HELICOPTER", "observation"); g = D[D.track_no == tn]
ax.plot(g.time_s, g.speed, color=COL["HELICOPTER"]); ax.axvspan(g.time_s.iloc[s0], g.time_s.iloc[s0 + n - 1], color="#b279a2", alpha=.25, label="hover"); ax.set_xlabel("time (s)"); ax.set_ylabel("speed (m/s)"); ax.legend(); ax.set_title("Helicopter speed through that track")
fig.suptitle("Hovering motion", fontsize=16, weight="bold", y=.98); save(fig, "07_hovering.png"); plt.show()

## Plot 8: disguises (acting as one class, truly another)

In [ ]:
# ---------------- Cell 11: disguises ----------------
pairs = [c for c in ["FIGHTER_AIRCRAFT", "DRONE", "SKIFF", "BOAT"]]
fig, axs = plt.subplots(2, 4, figsize=(21, 9))
for j, c in enumerate(pairs):
    d = D[(D.true_class == c) & D.disguised]; tns = d.track_no.unique()
    for r, tn in enumerate(tns[:2]):
        g = d[d.track_no == tn]; ax = axs[r, j]; ch = (g.acting_class != c).values
        ax.plot(g.time_s, g.speed, color=COL[c], lw=1.6)
        ax.axvspan(g.time_s[ch].min(), g.time_s[ch].max(), color=COL[g.acting_class[ch].iloc[0]], alpha=.18, label="acting as " + SHORT[g.acting_class[ch].iloc[0]])
        ax.axvline(g.time_s[ch].max(), color="k", ls="--", lw=1); ax.text(g.time_s[ch].max(), ax.get_ylim()[1]*.92, " breakout", fontsize=8)
        ax.set_title(f"True {SHORT[c]}" if r == 0 else "", color=COL[c]); ax.set_xlabel("time (s)"); ax.set_ylabel("speed (m/s)"); ax.legend(fontsize=8, loc="lower right")
fig.suptitle("Disguises: the target behaves like another class until the breakout", fontsize=15, weight="bold"); plt.tight_layout(); save(fig, "08_disguises_examples.png")
plt.show()
fig, axs = plt.subplots(1, 2, figsize=(14, 5.5))
dd = D[D.disguised]; s = dd.assign(dis=dd.acting_class != dd.true_class).groupby("true_class").dis.mean().reindex(pairs)
axs[0].bar([SHORT[c] for c in pairs], s*100, color=[COL[c] for c in pairs], label="% of packets while disguised"); axs[0].bar([SHORT[c] for c in pairs], (1 - s)*100, bottom=s*100, color="#dddddd", label="% after breakout")
axs[0].set_ylabel("% of the disguised tracks' packets"); axs[0].legend(); axs[0].set_title("Time disguised vs revealed")
bo = dd[dd.acting_class == dd.true_class].groupby("track_no").time_s.min(); axs[1].hist(bo, bins=25, color="#444"); axs[1].set_xlabel("time of breakout from track start (s)"); axs[1].set_title("When do disguises break out?")
plt.tight_layout(); save(fig, "08b_disguise_stats.png"); plt.show()

## Plot 9: data health checks

In [ ]:
# ---------------- Cell 12: data health ----------------
fig, axs = plt.subplots(1, 3, figsize=(20, 5.5))
dt = D.groupby("track_no").time_s.diff().dropna(); axs[0].hist(dt, bins=[2.45, 2.55], color="#2a9d8f"); axs[0].set_title(f"Time step between packets: {dt.min():.1f} to {dt.max():.1f} s (always 2.5)"); axs[0].set_xlabel("seconds")
L = D.groupby(["true_class", "track_no"]).size().reset_index(name="n"); data = [L[L.true_class == c].n for c in CLASSES]
bp = axs[1].boxplot(data, patch_artist=True, showfliers=False); axs[1].set_xticklabels([SHORT[c] for c in CLASSES])
for p, c in zip(bp["boxes"], CLASSES): p.set_facecolor(COL[c])
plt.setp(axs[1].get_xticklabels(), rotation=35, ha="right"); axs[1].set_ylabel("packets per track"); axs[1].set_title("Track length (x 2.5 s = duration)")
P = D[["pos_x", "pos_y", "pos_z"]].to_numpy(); V = D[["vel_x", "vel_y", "vel_z"]].to_numpy(); err = []
for tn, g in D.groupby("track_no"):
    p = g[["pos_x", "pos_y", "pos_z"]].to_numpy(); v = g[["vel_x", "vel_y", "vel_z"]].to_numpy(); vm = .5*(v[1:] + v[:-1])
    err.append((g.true_class.iloc[0], np.median(np.linalg.norm(np.diff(p, axis=0)/2.5 - vm, axis=1)/np.maximum(np.linalg.norm(vm, axis=1), 1))*100))
E = pd.DataFrame(err, columns=["c", "e"]); axs[2].bar([SHORT[c] for c in CLASSES], [E[E.c == c].e.median() for c in CLASSES], color=[COL[c] for c in CLASSES]); plt.setp(axs[2].get_xticklabels(), rotation=35, ha="right")
axs[2].set_ylabel("median % mismatch"); axs[2].set_title("Does velocity agree with the position changes?")
plt.tight_layout(); save(fig, "09_data_health.png"); plt.show()
print("\nAll files are in:", os.path.abspath(OUT_DIR))